# S1 · Reward models: Bradley–Terry, training, bias audit, process labels

Study path stage 1 of 7 · Chapter 10 · Day 16 · **DGX Spark GPU first**

A **reward model** is a network $r_\phi(x, y)$ that reads a prompt $x$ and a response $y$ and returns
one real number. It is trained on comparisons, "for prompt $x$, response $y_w$ was preferred to
$y_l$", and then scores responses nobody has judged. [Day 15 · 04](../day-15/04_preference_alignment_end_to_end.ipynb)
Part C trained one for you. In this lab you write the key lines yourself and then ask harder questions
of the result: what does the gradient do, what does label noise do to the learned weights, which hidden
state becomes the score, how certain is a held-out accuracy, does the score track length, and what a
per-step label can see that a per-answer label cannot. By the end you can build, train, measure and
audit a reward model, and say precisely what each number does and doesn't establish.

| Part | Question | Discussions in the [learning note](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
|---|---|---|
| A · Bradley–Terry | How does a score gap become a loss, and what gradient does each pair send? | D4, D7, D9, D10 |
| B · A linear reward model with a known truth | Does training recover the true reward? What do random and systematic label noise do to it? | D3, D5, D6, D8 |
| C · From a sequence to one number | Which position's hidden state becomes the score, and why that one? | D15, D16, D28, D30 |
| D · Train a Qwen3 reward model | How much does training move held-out accuracy, with what uncertainty? Is the score calibrated? | D17, D27, D29 |
| E · Length audit | Does the score follow answer length? Is accuracy the same when the chosen answer is shorter? | D2, D3, D15, D18 |
| F · Outcome and process labels | What does a per-step label locate that a per-answer label cannot? | D37, D38 |

### How the exercises work

1. **Read the task**, then write your lines where you see `...` in the exercise cell (usually one to
   three lines, never a large block).
2. **Run the check.** It tests your code on small cases with known answers and prints ✅ or ❌ with a
   hint that names the likely mistake. Nothing crashes if you haven't finished; you'll see ⏳ instead.
3. **Compare with the reference**, which follows each check and then uses the function on the toy
   problem or on Qwen3-0.6B.

Later cells use **your** version when its check passed, and the reference otherwise
(`mine_or_ref(globals(), …)` prints which), so one stuck exercise never blocks the rest. Prediction
cells ask you to commit to an answer before the reference computes it.

**Data.** Anthropic [HH-RLHF](https://huggingface.co/datasets/Anthropic/hh-rlhf) `helpful-base`
(MIT; Bai et al. 2022): one human comparison per pair. The `harmless-base` subset isn't used. The
linear-reward data (Part B) are synthetic draws from a stated model, and the arithmetic solutions
(Part F) are authored for this notebook with deliberate, labelled errors.

**Environment.** The `Python (DGX Spark Native)` kernel with CUDA, and Qwen3-0.6B plus HH-RLHF in
the local Hugging Face cache; nothing is downloaded. Checks and references come from the tested
module `dongxi_llms.study_path.s1_reward_models`; Qwen3 plumbing from `dongxi_llms.study_path.qwen`
(printed with `inspect.getsource` where first used).

**Run time.** About 5 minutes on the Spark once the GPU is free: Parts A–C and F take seconds, the
reward-model training in Part D about 3.5 minutes. Peak GPU memory is about 20 GiB (printed at the end):
float32 weights, gradients and AdamW state for 0.6 B parameters take about 10 GiB before activations.

**Scale and limits.** The toy parts use a known true reward, so recovery can be checked exactly.
The Qwen3 reward model is trained once (one seed, 1,200 pairs, one pass), so its numbers describe this
run; held-out comparisons carry binomial standard errors, and differences within about two standard
errors are not claimed. The process-label part uses an exact rule-based step checker and a declared
stand-in for a process reward model, not a trained one. RL terms are defined in the
[glossary](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md#-rl-terms-in-the-llm-setting-glossary-d39).

In [ ]:
import inspect, math, random, sys, time
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(ROOT / "src"))               # the course package, wherever this copy lives
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F
from dongxi_llms.study_path import s1_reward_models as s1
from dongxi_llms.study_path import qwen as Q
from dongxi_llms.study_path.common import mine_or_ref

assert torch.cuda.is_available(), "Use the Python (DGX Spark Native) kernel: this notebook is GPU first."
DEVICE = "cuda"
random.seed(0); np.random.seed(0); torch.manual_seed(0)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.family": "DejaVu Sans",
                     "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GRAY, GREEN, RED = "#2563EB", "#EA580C", "#94A3B8", "#16A34A", "#DC2626"
T_START = time.time()
torch.cuda.reset_peak_memory_stats()
print(f"torch {torch.__version__} | {torch.cuda.get_device_name()}")

---

# Part A · Bradley–Terry: the loss and its gradient

A preference dataset is a list of triples $(x, y_w, y_l)$: prompt, preferred ("chosen") response,
and dispreferred ("rejected") response. The reward model gives each response a score, and the
**Bradley–Terry model** turns two scores into a probability that depends only on their difference,
the **score gap** $\Delta = r_\phi(x, y_w) - r_\phi(x, y_l)$:

$$P(y_w \succ y_l \mid x) = \sigma(\Delta) = \frac{1}{1 + e^{-\Delta}}.$$

Training maximizes the probability of the labels we observed. Taking $-\log$ and averaging over $N$
pairs gives the **pairwise loss**, which is binary cross-entropy with target 1 on the logit $\Delta$ (D9):

$$\mathcal L(\phi) = -\frac{1}{N}\sum_{i=1}^{N} \log \sigma(\Delta_i).$$

### Prediction

## Exercise 1 · the Bradley–Terry probability and loss

Write `my_bt_prob(gap)`, the probability that the chosen response wins, and `my_bt_loss(r_chosen, r_rejected)`,
the mean pairwise loss over a batch of pairs (both arguments have shape `[N]`). One line each.

Compute the loss **stably**: the check feeds a confidently wrong pair with gap −200, where
$\sigma(-200)$ rounds to 0 in float32.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_bt_prob(gap):
    return ...            # P(chosen ≻ rejected) for each gap

def my_bt_loss(r_chosen, r_rejected):
    return ...            # mean over pairs of −log σ(r_chosen − r_rejected), computed stably

In [ ]:
ex1_ok = s1.check_bt(globals().get("my_bt_prob"), globals().get("my_bt_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def bt_prob_ref(gap):
    return torch.sigmoid(gap)

def bt_loss_ref(r_chosen, r_rejected):
    return -F.logsigmoid(r_chosen - r_rejected).mean()

s1.check_bt(bt_prob_ref, bt_loss_ref, strict=True)
bt_prob = mine_or_ref(globals(), "my_bt_prob", ex1_ok, bt_prob_ref)
bt_loss = mine_or_ref(globals(), "my_bt_loss", ex1_ok, bt_loss_ref)

far = torch.tensor([-200.0])
print(f"gap −200: −log(sigmoid(gap)) = {(-torch.log(torch.sigmoid(far))).item()} | "
      f"−logsigmoid(gap) = {bt_loss(far, torch.zeros(1)).item():.1f}\n")
rc, rr = torch.tensor([0.0, 1.0, 2.0, 5.0]), torch.tensor([2.0, 1.0, 1.0, 1.0])
for name, (a, b) in {"scores": (rc, rr), "+10 to both": (rc + 10, rr + 10), "both × 2": (2 * rc, 2 * rr)}.items():
    print(f"{name:12s} loss {bt_loss(a, b).item():.4f} | P(chosen wins) "
          + " ".join(f"{v:.3f}" for v in bt_prob(a - b).tolist()))
assert torch.allclose(bt_loss(rc + 10, rr + 10), bt_loss(rc, rr))

### Reference explanation

### Reference answer and mechanism

`F.logsigmoid` computes $\log\sigma(\Delta) = -\log(1 + e^{-\Delta})$ without forming $\sigma(\Delta)$ first.
The naive version rounds $\sigma(-200)$ to 0 and returns `inf`; the stable one returns 200, the
correct value. One confidently wrong pair would otherwise turn the whole batch's loss into `inf`.

The three rows show the two symmetries of the scores (Chapter 10 §10.5):

- **Shift.** Adding 10 to both scores leaves every gap, every probability and the loss unchanged.
  The score has no meaningful zero; only differences between responses to the *same* prompt carry
  information (D15).
- **Scale.** Multiplying both scores by 2 doubles every gap, so probabilities move away from 0.5 and the
  loss changes: here it rises from 0.788 to 1.210, because the wrongly ranked pair (gap −2 → −4) costs
  more than the two correctly ranked pairs gain. The scale of the scores is not free: it encodes how
  confident the model is. Part B shows that label noise sets it.

### The gradient each pair sends

Training moves the scores along $-\partial \mathcal L / \partial r$. For one pair, write
$\ell = -\log\sigma(\Delta)$. The derivative of the sigmoid is $\sigma'(\Delta) = \sigma(\Delta)\,(1 - \sigma(\Delta))$, so

$$\frac{d\ell}{d\Delta} = -\frac{\sigma'(\Delta)}{\sigma(\Delta)} = -\frac{\sigma(\Delta)(1-\sigma(\Delta))}{\sigma(\Delta)} = -\bigl(1 - \sigma(\Delta)\bigr).$$

Because $\Delta = r_w - r_l$, the chain rule gives $\partial\ell/\partial r_w = d\ell/d\Delta \cdot 1$ and
$\partial\ell/\partial r_l = d\ell/d\Delta \cdot (-1)$.

### Prediction

## Exercise 2 · the gradient by hand

Write the per-pair gradients of the summed loss $\sum_i \ell_i$ with respect to the chosen and the
rejected scores, using the formula above (no autograd). The check compares them with
`torch.autograd` on five pairs.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_bt_grads(r_chosen, r_rejected):
    gap = r_chosen - r_rejected
    g_chosen = ...              # ∂ℓ_i / ∂r_chosen,i
    g_rejected = ...            # ∂ℓ_i / ∂r_rejected,i
    return g_chosen, g_rejected

In [ ]:
ex2_ok = s1.check_bt_grads(globals().get("my_bt_grads"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def bt_grads_ref(r_chosen, r_rejected):
    gap = r_chosen - r_rejected
    return -(1 - torch.sigmoid(gap)), 1 - torch.sigmoid(gap)

s1.check_bt_grads(bt_grads_ref, strict=True)
bt_grads = mine_or_ref(globals(), "my_bt_grads", ex2_ok, bt_grads_ref)

rc = torch.tensor([-3.0, 0.0, 1.0, 4.0], requires_grad=True)
rr = torch.zeros(4)
(-F.logsigmoid(rc - rr)).sum().backward()
g_c, g_r = bt_grads(rc.detach(), rr)
for gap, a, h in zip((rc - rr).tolist(), rc.grad.tolist(), g_c.tolist()):
    print(f"gap {gap:+.0f}: autograd {a:+.4f} | by hand {h:+.4f}")
torch.testing.assert_close(g_c, rc.grad)
torch.testing.assert_close(g_r, -g_c)

### Reference explanation

### Reference answer and mechanism

The gradient on the chosen score is $-(1 - \sigma(\Delta))$ and on the rejected score $+(1 - \sigma(\Delta))$:
equal size, opposite sign. A gradient step therefore raises the chosen score and lowers the rejected one
by the same amount, which changes the gap by twice that amount and changes nothing else.

The size $1 - \sigma(\Delta)$ is the model's probability of the *wrong* order. A pair ranked wrongly
(gap −3) sends a gradient of magnitude 0.95; a pair ranked right by a wide margin (gap +4) sends 0.018.
Training concentrates on the pairs the model doesn't yet get right, and the gradient never reaches 0
for a finite gap: with clean, separable labels the scores would keep growing. Part B shows what stops
them with noisy labels.

In [ ]:
g = torch.linspace(-6, 6, 241, dtype=torch.float64)
pts = torch.linspace(-5, 5, 11, dtype=torch.float64, requires_grad=True)
(-F.logsigmoid(pts)).sum().backward()                          # autograd at 11 gaps (r_rejected = 0)
g_c, g_r = bt_grads(g, torch.zeros_like(g))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(g, g_c, color=RED, label="∂ℓ/∂r_chosen = −(1 − σ(Δ))  (by hand)")
axes[0].plot(g, g_r, color=BLUE, label="∂ℓ/∂r_rejected = +(1 − σ(Δ))  (by hand)")
axes[0].scatter(pts.detach(), pts.grad, color="black", s=16, zorder=3, label="autograd, ∂ℓ/∂r_chosen")
axes[0].axhline(0, color=GRAY, lw=0.8); axes[0].axvline(0, color=GRAY, lw=0.8)
axes[0].set_xlabel("score gap Δ = r_chosen − r_rejected"); axes[0].set_ylabel("gradient of one pair's loss")
axes[0].set_title("Wrongly ranked pairs get the largest push"); axes[0].legend(fontsize=8)
for k, c in ((0.5, GRAY), (1.0, BLUE), (2.0, ORANGE)):
    axes[1].plot(g, bt_prob(k * g), color=c, label=f"all scores × {k:g}: σ({k:g}·Δ)")
axes[1].scatter([-2, 0, 1, 4], bt_prob(torch.tensor([-2.0, 0, 1, 4]) + 10 - 10), color="black", s=16, zorder=3,
                label="all scores + 10: unchanged")
axes[1].set_xlabel("score gap Δ at scale 1"); axes[1].set_ylabel("P(chosen ≻ rejected)")
axes[1].set_title("A shift changes nothing; a scale changes confidence"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Bradley–Terry gradient by hand against autograd, and the effect of shifting or scaling all scores](../figures/chapter-10/study-path-s1_reward_models-01.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part B · A linear reward model with a known truth

On real data we never see the true reward, so we can't check whether training recovered it. A
simulation where we choose the truth can. Each response gets a feature vector
$f = (\text{helpfulness}, \text{length}) \in \mathbb R^2$, both standardized (mean 0, standard deviation 1),
and the **true reward** is linear, $r^*(f) = w^{*\top} f$ with $w^* = (2.0,\ 0.5)$: helpfulness matters
four times as much as length. A judge compares responses $A$ and $B$ and, when it is noise-free,
says "$A$ wins" with probability $\sigma(r^*(f_A) - r^*(f_B))$. So Bradley–Terry is the true model here,
and a linear reward model $r_w(f) = w^\top f$ can represent the truth exactly.

The generator below draws the pairs, then optionally corrupts a fraction $\varepsilon$ of the labels in
one of two ways (used from Exercise 4 on).

In [ ]:
print(inspect.getsource(s1.make_linear_pairs))
clean = s1.make_linear_pairs(4000, seed=1)
print("chosen features fc:", tuple(clean["fc"].shape), "| rejected features fr:", tuple(clean["fr"].shape))
print("first pair: chosen", clean["fc"][0].numpy().round(2), "rejected", clean["fr"][0].numpy().round(2))

### Prediction

## Exercise 3 · fit a linear reward model

Write the loss line of a full-batch gradient-descent fit. `fc` and `fr` are `[N, 2]` feature
matrices of the chosen and rejected responses, and the reward model is $r_w(f) = w^\top f$. Use your
`bt_loss` (it is in scope). One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_fit_linear_rm(fc, fr, steps=300, lr=1.0):
    w = torch.zeros(fc.shape[1], dtype=fc.dtype, requires_grad=True)
    history = []
    for _ in range(steps):
        loss = ...                                # Bradley–Terry loss of the linear scores
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            w.grad.zero_()
        history.append(loss.item())
    return w.detach(), history

In [ ]:
ex3_ok = s1.check_fit_linear_rm(globals().get("my_fit_linear_rm"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def fit_linear_rm_ref(fc, fr, steps=300, lr=1.0):
    w = torch.zeros(fc.shape[1], dtype=fc.dtype, requires_grad=True)
    history = []
    for _ in range(steps):
        loss = bt_loss_ref(fc @ w, fr @ w)
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            w.grad.zero_()
        history.append(loss.item())
    return w.detach(), history

s1.check_fit_linear_rm(fit_linear_rm_ref, strict=True)
fit_linear_rm = mine_or_ref(globals(), "my_fit_linear_rm", ex3_ok, fit_linear_rm_ref)

w_clean, hist = fit_linear_rm(clean["fc"], clean["fr"])
print(f"true w* = {clean['w_true'].tolist()} | fitted on 4,000 clean pairs: {w_clean.numpy().round(3).tolist()}")
print(f"loss {hist[0]:.4f} (w = 0, i.e. log 2) → {hist[-1]:.4f} after {len(hist)} steps")

### Reference explanation

### Reference answer and mechanism

With 4,000 clean labels the fit lands close to $w^* = (2.0, 0.5)$, within sampling error. Two facts
make this work. First, the model family contains the truth. Second, the scale of $w$ is identified:
the labels were drawn from $\sigma(w^{*\top}(f_A - f_B))$, so a larger $w$ would be over-confident and a
smaller one under-confident, and the likelihood is maximized near $w^*$. This is D6's point in
numbers: each pair has one label, but thousands of single labels on different pairs pin down one
shared scoring function.

The loss starts at $\log 2 = 0.693$ (all scores 0, every probability 0.5) and doesn't go to 0. Even
the true $w^*$ has a positive loss, because the labels themselves are random: a pair with a small true gap
is close to a coin flip.

### Two kinds of label noise

Real judges make mistakes (D3). We model two kinds, each applied to a fraction $\varepsilon$ of the pairs:

- **Random flips** (`judge="random"`): with probability $\varepsilon$ the label is reversed, whatever the
  responses look like. This models an inattentive judge.
- **Length-biased judging** (`judge="length"`): with probability $\varepsilon$ the judge ignores
  quality and picks the longer response. This models a systematic bias.

For random flips the probability of the label "$A$ wins" can be written in closed form as a function of
the true gap $\Delta^* = r^*(f_A) - r^*(f_B)$: the clean label says $A$ with probability $\sigma(\Delta^*)$, and
a flip turns a clean "$B$" into "$A$".

### Prediction

## Exercise 4 · the probability of a noisy label

Write $P(\text{label} = A \mid \Delta^*, \varepsilon)$ for random flips. Combine the two ways the label can end up "$A$":
clean "$A$" and not flipped, or clean "$B$" and flipped. One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_noisy_label_prob(gap_true, eps):
    return ...            # P(label says A wins) after flipping each clean label with probability eps

In [ ]:
ex4_ok = s1.check_noisy_label_prob(globals().get("my_noisy_label_prob"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def noisy_label_prob_ref(gap_true, eps):
    return (1 - eps) * torch.sigmoid(gap_true) + eps * (1 - torch.sigmoid(gap_true))

s1.check_noisy_label_prob(noisy_label_prob_ref, strict=True)
noisy_label_prob = mine_or_ref(globals(), "my_noisy_label_prob", ex4_ok, noisy_label_prob_ref)

big = s1.make_linear_pairs(200_000, seed=7, eps=0.2)          # many pairs, to compare with the formula
edges = torch.tensor([-8.0, -4, -2, -1, 0, 1, 2, 4, 8])
which = torch.bucketize(big["gap_true"], edges)
print("true gap bin      observed P(label = A)   formula at bin mean")
for b in range(1, len(edges)):
    m = which == b
    observed = big["a_wins"][m].double().mean().item()
    formula = noisy_label_prob(big["gap_true"][m], 0.2).mean().item()
    print(f"[{edges[b-1]:+.0f}, {edges[b]:+.0f})        {observed:.3f}                   {formula:.3f}")

### Reference explanation

### Reference answer and mechanism

$P(A) = (1-\varepsilon)\,\sigma(\Delta^*) + \varepsilon\,(1-\sigma(\Delta^*)) = \varepsilon + (1-2\varepsilon)\,\sigma(\Delta^*)$.
The empirical frequencies on 200,000 simulated pairs match the formula in every bin.

The important feature is the **floor and ceiling**: however large the true gap, the label says "$A$"
at most $1-\varepsilon$ of the time (0.8 for $\varepsilon = 0.2$) and at least $\varepsilon$. A
Bradley–Terry model $\sigma(w^\top(f_A - f_B))$ goes all the way to 0 and 1 and has no such plateau, so
with noisy labels the model is misspecified. Keep this in mind for the prediction after the next
exercise.

### Calibration

A model's probabilities are **calibrated** when, among all pairs where it predicts "$A$ wins with
probability 0.8", $A$ wins about 80% of the time (Chapter 10 §10.7). We measure this by **binning**:
group the predictions $p_i$ into $B$ equal-width bins, and in each bin $b$ compare the mean prediction
$\text{conf}_b$ with the observed frequency of label 1, $\text{freq}_b$. The **expected calibration error**
weights each bin by its share of the $N$ predictions:

$$\text{ECE} = \sum_{b} \frac{n_b}{N}\,\bigl|\text{conf}_b - \text{freq}_b\bigr|.$$

### Prediction

## Exercise 5 · a reliability table and the ECE

Fill in the per-bin mean prediction, the per-bin frequency of label 1, and the ECE. Three lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_reliability(p, y, n_bins=10):
    p, y = p.detach().double().flatten(), y.detach().double().flatten()
    bins = torch.clamp((p * n_bins).long(), max=n_bins - 1)       # bin index of each prediction
    conf, freq, count = [], [], []
    for b in range(n_bins):
        m = bins == b
        if m.any():
            conf.append(...)              # mean predicted probability in bin b (a float)
            freq.append(...)              # fraction of label 1 in bin b (a float)
            count.append(int(m.sum()))
    ece = ...                             # Σ_b (count_b / N) · |conf_b − freq_b|
    return {"conf": conf, "freq": freq, "count": count, "ece": ece}

In [ ]:
ex5_ok = s1.check_reliability(globals().get("my_reliability"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def reliability_ref(p, y, n_bins=10):
    p, y = p.detach().double().flatten(), y.detach().double().flatten()
    bins = torch.clamp((p * n_bins).long(), max=n_bins - 1)
    conf, freq, count = [], [], []
    for b in range(n_bins):
        m = bins == b
        if m.any():
            conf.append(p[m].mean().item())
            freq.append(y[m].mean().item())
            count.append(int(m.sum()))
    ece = sum(n / len(p) * abs(c - f) for c, f, n in zip(conf, freq, count))
    return {"conf": conf, "freq": freq, "count": count, "ece": ece}

s1.check_reliability(reliability_ref, strict=True)
reliability = mine_or_ref(globals(), "my_reliability", ex5_ok, reliability_ref)

test = s1.make_linear_pairs(20_000, seed=99)                     # held-out pairs, clean labels
p_clean_fit = torch.sigmoid((test["fa"] - test["fb"]) @ w_clean)
rel = reliability(p_clean_fit, test["a_wins"])
print(f"clean-label fit on 20,000 held-out clean pairs: ECE = {rel['ece']:.4f}")

### Reference explanation

### Reference answer and mechanism

`p[m].mean()` and `y[m].mean()` are the two coordinates of a point on a reliability diagram; a
calibrated model's points lie on the diagonal. The weights $n_b/N$ keep a nearly empty bin from
dominating the summary, but the ECE still depends on the bin count and hides which bins are off, so
the figure below shows the bins themselves.

The model fitted to clean labels has an ECE below 0.01 on held-out pairs: the model family contains the
true probability model, and 4,000 pairs estimate it well. Next, the noisy judges.

### Prediction

## Predict 1 · what does 20% label noise do?

Refit the linear reward model on 4,000 pairs where **20% of the labels were flipped at random**.
Predict, compared with the clean fit $(2.0, 0.5)$:

1. the **direction** of $w$ (the ratio $w_\text{length}/w_\text{help}$, truly 0.25),
2. the **scale** $\|w\|$,
3. **ranking agreement** with the true reward on held-out pairs (how often $\text{sign}(w^\top \Delta f) = \text{sign}(w^{*\top} \Delta f)$),
4. **calibration**: against the noisy labels, and against clean labels.

Then the same for the **length-biased** judge at $\varepsilon = 0.2$. Which of the two noise types is worse for
ranking, even though it changes fewer labels?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
def agreement(w, pairs):
    """How often the fitted reward orders a held-out pair the same way as the true reward."""
    return (((pairs["fa"] - pairs["fb"]) @ w > 0) == (pairs["gap_true"] > 0)).double().mean().item()

EPS = [0.0, 0.1, 0.2, 0.3, 0.4]
sweep = {}
print("judge    ε     labels changed   w_help   w_length   ratio   ‖w‖     agreement with r*")
for judge in ("random", "length"):
    for eps in EPS:
        d = s1.make_linear_pairs(4000, seed=1, eps=eps, judge=judge)
        w, _ = fit_linear_rm(d["fc"], d["fr"])
        changed = (d["a_wins"] != d["a_wins_clean"]).double().mean().item()
        sweep[judge, eps] = (w, agreement(w, test))
        print(f"{judge:7s}  {eps:.1f}   {changed:6.1%}          {w[0]:.3f}    {w[1]:.3f}      {w[1] / w[0]:.3f}  "
              f"{w.norm():.3f}   {sweep[judge, eps][1]:.4f}")

w20 = sweep["random", 0.2][0]
noisy_test = s1.make_linear_pairs(20_000, seed=99, eps=0.2)       # same held-out pairs, labels flipped at 20%
p20 = torch.sigmoid((noisy_test["fa"] - noisy_test["fb"]) @ w20)
rel_noisy, rel_true = reliability(p20, noisy_test["a_wins"]), reliability(p20, noisy_test["a_wins_clean"])
print(f"\nε = 0.2 random-flip fit: ECE {rel_noisy['ece']:.3f} against noisy labels, "
      f"{rel_true['ece']:.3f} against clean labels")
assert abs(float(w20[1] / w20[0]) - 0.25) < 0.05 and sweep["random", 0.2][1] > 0.99
assert sweep["length", 0.2][1] < sweep["random", 0.2][1]

### Reference explanation

### Reference answer and mechanism

**Random flips shrink the scale and keep the direction.** At $\varepsilon = 0.2$ the fitted weights fall to
about $(0.72, 0.18)$, roughly a third of the clean scale, but the ratio stays at 0.25 and the fitted reward
orders held-out pairs the same way as the true reward 99.9% of the time. The flips point in no
consistent direction, so they cancel in the gradient; what they change is how confident the model can
be. The noisy labels never exceed $1-\varepsilon = 0.8$, so the best Bradley–Terry fit flattens its
sigmoid to stay near the plateau, and the flattening is stronger than the naive factor $1 - 2\varepsilon = 0.6$
because $\sigma$ can't bend to a floor and ceiling.

**Calibration depends on which labels you ask about.** Against the noisy labels it was trained on, the
fit is roughly calibrated (ECE about 0.04, not 0, because of the misspecified plateau). Against the clean
preference it is **under-confident** (ECE about 0.15): it says 0.7 where the clean judge says 0.9.

**Systematic bias changes the direction.** The length-biased judge at $\varepsilon = 0.2$ changes only about
8% of the labels (it only matters when the clean label already picked the shorter response), yet the
ratio $w_\text{length}/w_\text{help}$ doubles from 0.25 to about 0.53 and ranking agreement drops to about 92%.
The model has learned the judge's habit as part of the reward. More data won't fix this: the bias is in
the label distribution, not in the sample (D3: random noise averages out, systematic bias is learned).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for judge, ls in (("random", "-"), ("length", "--")):
    axes[0].plot(EPS, [sweep[judge, e][0][0].item() for e in EPS], ls, marker="o", color=BLUE,
                 label=f"w_help, {judge} judge")
    axes[0].plot(EPS, [sweep[judge, e][0][1].item() for e in EPS], ls, marker="o", color=ORANGE,
                 label=f"w_length, {judge} judge")
axes[0].axhline(2.0, color=BLUE, lw=0.6, alpha=0.5); axes[0].axhline(0.5, color=ORANGE, lw=0.6, alpha=0.5)
axes[0].set_xlabel("corrupted fraction ε"); axes[0].set_ylabel("fitted weight")
axes[0].set_title("Fitted weights (thin lines: true w*)"); axes[0].legend(fontsize=7)

gt = torch.linspace(-8, 8, 200, dtype=torch.float64)
centers = torch.linspace(-7, 7, 15, dtype=torch.float64)
which = torch.bucketize(big["gap_true"], centers - 0.5)
emp = [big["a_wins"][which == i + 1].double().mean().item() for i in range(len(centers))]
axes[1].scatter(centers, emp, color="black", s=14, zorder=3, label="observed, ε = 0.2 (200k pairs)")
axes[1].plot(gt, noisy_label_prob(gt, 0.2), color=RED, label="ε + (1 − 2ε)·σ(Δ*)  (Exercise 4)")
axes[1].plot(gt, torch.sigmoid(gt * (w20.norm() / torch.tensor([2.0, 0.5], dtype=torch.float64).norm())),
             color=BLUE, label="fitted BT model σ(c·Δ*)")
axes[1].plot(gt, torch.sigmoid(gt), color=GRAY, ls=":", label="clean σ(Δ*)")
axes[1].set_xlabel("true gap Δ*"); axes[1].set_ylabel("P(label says A)")
axes[1].set_title("Noise adds a floor and ceiling; BT flattens"); axes[1].legend(fontsize=7)

for rel_, name, c in ((rel_noisy, "against noisy labels", ORANGE), (rel_true, "against clean labels", BLUE)):
    axes[2].plot(rel_["conf"], rel_["freq"], marker="o", color=c, label=f"{name}: ECE {rel_['ece']:.3f}")
axes[2].plot([0, 1], [0, 1], color=GRAY, lw=0.8, ls=":")
axes[2].set_xlabel("predicted P(A wins), ε = 0.2 fit"); axes[2].set_ylabel("observed frequency")
axes[2].set_title("Reliability on 20,000 held-out pairs"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Fitted reward weights under random and length-biased label noise, the noisy-label curve, and reliability](../figures/chapter-10/study-path-s1_reward_models-02.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

The middle panel draws the fitted model along the true-gap axis. This is exact here because random flips
keep the fitted $w$ parallel to $w^*$, so $w^\top \Delta f = c\,\Delta^*$ with $c = \|w\|/\|w^*\|$.

---

# Part C · From a sequence to one number

A real reward model replaces the linear features with a transformer. Qwen3-0.6B maps a token sequence
of length $T$ to hidden states $h_1, \dots, h_T \in \mathbb R^{1024}$. A causal language model ends
in `lm_head`, a $1024 \to 151{,}936$ linear layer giving one logit per vocabulary token. The reward model
instead ends in `score`, a $1024 \to 1$ linear layer $w^\top h_t$, which gives one number at **every**
position. The returned reward is the number at one position: the last real token, here `<|im_end|>`
(D28). Hugging Face's `AutoModelForSequenceClassification` with `num_labels=1` performs this head
swap at load time (D30). The cell below loads it and prints the load report.

In [ ]:
print(inspect.getsource(Q.load_reward_model))
tok = Q.load_tokenizer()
PAD, IM_END = tok.pad_token_id, tok.convert_tokens_to_ids("<|im_end|>")
print(f"pad token {tok.pad_token!r} = {PAD} | <|im_end|> = {IM_END}\n")
torch.manual_seed(0)                          # the new score head is randomly initialized: fix its draw
rm = Q.load_reward_model(tok, dtype=torch.float32)   # fp32 master weights; Hugging Face prints its load report

def bf16():
    """Mixed precision: matrix multiplies run in bfloat16, the weights and their updates stay float32."""
    return torch.autocast("cuda", dtype=torch.bfloat16)

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors import safe_open
with safe_open(hf_hub_download(Q.MODEL, "model.safetensors"), "pt") as f:
    ckpt = {k: tuple(f.get_slice(k).get_shape()) for k in f.keys()}
model_keys = {k: tuple(v.shape) for k, v in rm.state_dict().items()}
print("in the checkpoint, not in the reward model (discarded):", {k: ckpt[k] for k in ckpt.keys() - model_keys.keys()})
print("in the reward model, not in the checkpoint (new, random):", {k: model_keys[k] for k in model_keys.keys() - ckpt.keys()})
print(f"shared tensors: {len(ckpt.keys() & model_keys.keys())} | parameters: {sum(p.numel() for p in rm.parameters()) / 1e6:.1f} M")
print("rm.score =", rm.score, "| has lm_head:", hasattr(rm, "lm_head"))

print("\n" + inspect.getsource(Q.to_pair))
train_pairs, test_pairs = Q.hh_pairs("train"), Q.hh_pairs("test")
print(f"HH-RLHF helpful-base clean pairs: {len(train_pairs)} train, {len(test_pairs)} test")

The load report and the key comparison say the same thing: the checkpoint's `lm_head.weight`
($151{,}936 \times 1024$) is **UNEXPECTED** and dropped, and `score.weight` ($1 \times 1024$) is **MISSING** and
newly initialized. All the shared tensors (counted above), the embeddings and 28 transformer layers,
are the pretrained Qwen3 body, loaded unchanged. With no vocabulary layer there is no next-token distribution, so this model
can't generate text; it returns one number per sequence (D30). `Q.to_pair` turns an HH record into a
`(prompt, chosen, rejected)` triple when the two transcripts differ only in the last assistant turn
(Day 15 · 04 Part A).

### Which position?

A batch holds sequences of different lengths, so shorter rows are filled with the pad token
(`<|endoftext|>`, id 151643). Padding goes on the right or on the left. Hugging Face's rule finds the
**rightmost position whose token is not the pad token**, which is correct for both sides.

### Prediction

## Exercise 6 · the last non-pad position

Given `input_ids` of shape `[B, T]` and the pad id, return a `[B]` tensor with the index of the last
real token in each row. It must work for right **and** left padding. Two lines: a 0/1 mask of non-pad
tokens, then the rightmost column where the mask is 1 (hint: multiply the mask by `torch.arange(T)`).

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_last_token_index(input_ids, pad_id):
    non_pad = ...             # [B, T]: 1 where the token is not padding, else 0 (integer tensor)
    return ...                # [B]: rightmost column with non_pad == 1

In [ ]:
ex6_ok = s1.check_last_token_index(globals().get("my_last_token_index"), pad_id=PAD)

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def last_token_index_ref(input_ids, pad_id):
    non_pad = (input_ids != pad_id).long()
    return (torch.arange(input_ids.shape[-1], device=input_ids.device) * non_pad).argmax(-1)

s1.check_last_token_index(last_token_index_ref, pad_id=PAD, strict=True)
last_token_index = mine_or_ref(globals(), "my_last_token_index", ex6_ok, last_token_index_ref)

pair = test_pairs[0]
texts = [Q.render(tok, pair["prompt"], pair["chosen"]), Q.render(tok, pair["prompt"], "Yes.")]
scores = {}
for mode in ("float32", "bf16 autocast"):              # float32 first: no rounding noise in the comparison
    for side in ("right", "left"):
        enc = tok(texts, return_tensors="pt", padding=True, padding_side=side).to(DEVICE)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=mode != "float32"):
            per_position = rm.score(rm.model(**enc).last_hidden_state).squeeze(-1).float()   # [B, T]
            returned = rm(**enc).logits.squeeze(-1).float()                                 # [B]
        idx = last_token_index(enc.input_ids, PAD)
        mine = per_position[torch.arange(len(texts)), idx]
        torch.testing.assert_close(mine, returned)
        scores[mode, side] = returned
        if mode == "float32":
            print(f"{side:5s} padding: input_ids {tuple(enc.input_ids.shape)} | per-position {tuple(per_position.shape)} "
                  f"| last index {idx.tolist()} = {[tok.decode(enc.input_ids[i, j]) for i, j in enumerate(idx.tolist())]}"
                  f" | scores {returned.cpu().numpy().round(4).tolist()}")
for mode in ("float32", "bf16 autocast"):
    d = (scores[mode, "right"] - scores[mode, "left"]).abs().max().item()
    print(f"{mode:14s} max |score(right pad) − score(left pad)| = {d:.5f}")

### Reference explanation

### Reference answer and mechanism

`(arange(T) * non_pad).argmax(-1)` multiplies each column index by 0 or 1, so the largest surviving index
is the rightmost real token. Your index reproduces the score Hugging Face returns, for both padding sides,
and the token there is `<|im_end|>` because `Q.render` ends every text on it.

The tempting `non_pad.sum(-1) - 1` counts real tokens; it equals the last index only under right
padding. Under left padding it points into the middle of the sequence. Two further failure modes from
D28: if the pad id were the same as `<|im_end|>`, the rule would skip the real end token, and without a
pad id Hugging Face refuses batches larger than 1.

In float32 the two padding sides give the same score to within $10^{-5}$. Padding tokens are masked out of
attention, and Qwen3's rotary position encoding makes attention depend only on position *differences*, so
shifting a sequence right by its left padding changes nothing. With bfloat16 matrix multiplies (the
autocast mode used for training and evaluation below) the same scores differ by about 0.008 here (and by
a few hundredths when the weights themselves are bfloat16): that is rounding noise, and it sets a floor on how precisely any single score can be read.

### Prediction

## Predict 2 · what does each position see?

Take two answers to the same question that differ in **one word in the middle**:
"Plant each clove two inches deep, pointy end up." and "… six inches deep, …". Compute the per-position
outputs $w^\top h_t$ for both. At which positions do the two sequences' outputs differ: everywhere, only at the
changed word, or from the changed word to the end? What does that imply about which position can serve as
the score of the whole answer?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
prompt = [{"role": "user", "content": "How deep should I plant garlic?"}]
a = Q.render(tok, prompt, "Plant each clove two inches deep, pointy end up.")
b = Q.render(tok, prompt, "Plant each clove six inches deep, pointy end up.")
ea, eb = tok([a], return_tensors="pt").to(DEVICE), tok([b], return_tensors="pt").to(DEVICE)
assert ea.input_ids.shape == eb.input_ids.shape
with torch.no_grad():
    pa = rm.score(rm.model(**ea).last_hidden_state).squeeze(-1)[0].float()
    pb = rm.score(rm.model(**eb).last_hidden_state).squeeze(-1)[0].float()
changed = (ea.input_ids[0] != eb.input_ids[0]).nonzero().flatten().tolist()
diff = (pa - pb).abs()
start = changed[0] - 3
for t in range(start, ea.input_ids.shape[1]):
    mark = "  ← changed token" if t in changed else ""
    print(f"position {t:2d} {tok.decode(ea.input_ids[0, t])!r:>12} | {tok.decode(eb.input_ids[0, t])!r:>12} "
          f"| |difference| {diff[t]:.4f}{mark}")
assert diff[:changed[0]].max() == 0 and (diff[changed[0]:] > 0).all()

### Reference explanation

### Reference answer and mechanism

The outputs are **identical, to the last bit, at every position before the changed word**, and differ at
the changed word and at **every** position after it. This is causal attention: $h_t$ is computed only from
tokens $1, \dots, t$, so a change at position $k$ can only affect $h_k, h_{k+1}, \dots, h_T$.

So the only position whose hidden state depends on the whole prompt and the whole answer is the last one.
Reading the score there is what makes it a score of the complete response. Reading it at position 0
would give the same number for every answer; reading it mid-answer would ignore everything after (the
truncation caveat of D27: with `max_length=320`, a longer conversation is cut on the right and its score
is read before the end of the answer). The untrained head's values themselves are meaningless; the
mechanism is in the pattern of zeros.

---

# Part D · Train a Qwen3 reward model

Day 15 · 04 trained on 3,000 pairs. Here we train on **1,200** pairs (one pass, batch 8, 150 steps) and
focus on measurement: how large is the improvement compared with its uncertainty, and is the model
calibrated? Training pairs come from `train_pairs[:20000]` (the RL prompts of later stages come from
`[20000:]`); evaluation uses the first 600 test pairs.

`Q.rm_scores` scores (prompt, answer) lists in batches under `torch.no_grad()`, so it serves evaluation.
It renders each conversation with `Q.render`, right-pads, truncates at 320 tokens and returns the pooled
score you reproduced in Exercise 6. The next cell also counts how many evaluation texts exceed 320
tokens: for those (about one in six in the reference run) the score is read before `<|im_end|>`, part-way
through the answer (D27). We keep the course's setting so train and evaluation texts match; cutting the
prompt from the left instead would keep every answer's end.

In [ ]:
print(inspect.getsource(Q.render)); print(inspect.getsource(Q.rm_scores))
EVAL = test_pairs[:600]
TRAIN = random.Random(1).sample(train_pairs[:20000], 1200)
n_tok = [len(tok(Q.render(tok, p["prompt"], a)).input_ids) for p in EVAL for a in (p["chosen"], p["rejected"])]
print(f"evaluation texts longer than 320 tokens (score read before <|im_end|>): {np.mean(np.array(n_tok) > 320):.1%}")

def eval_scores(model, pairs):
    """Held-out scores of the chosen and of the rejected answers, each [N] on the CPU."""
    model.eval()
    prompts = [p["prompt"] for p in pairs]
    with bf16():
        return (Q.rm_scores(model, tok, prompts, [p["chosen"] for p in pairs]).cpu(),
                Q.rm_scores(model, tok, prompts, [p["rejected"] for p in pairs]).cpu())

### Pair accuracy and its uncertainty

**Pair accuracy** is the fraction of held-out pairs where the chosen answer gets the higher score; a tie
counts as half. Each pair is one Bernoulli outcome, so with $N$ independent pairs the **standard error** of
the accuracy $a$ is $\sqrt{a(1-a)/N}$. For $N = 600$ and $a \approx 0.6$ that is 0.020, so two
accuracies need to differ by about 0.04–0.06 before the difference means anything.

### Prediction

## Exercise 7 · pair accuracy with a standard error

Return `(accuracy, standard_error)` as Python floats for score tensors `s_chosen`, `s_rejected` of shape `[N]`.
Count a tie as half a win. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_pair_accuracy(s_chosen, s_rejected):
    acc = ...                     # mean over pairs of 1[s_c > s_r] + 0.5·1[s_c == s_r], as a float
    se = ...                      # binomial standard error of acc
    return acc, se

In [ ]:
ex7_ok = s1.check_pair_accuracy(globals().get("my_pair_accuracy"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def pair_accuracy_ref(s_chosen, s_rejected):
    acc = ((s_chosen > s_rejected).double() + 0.5 * (s_chosen == s_rejected).double()).mean().item()
    return acc, math.sqrt(acc * (1 - acc) / len(s_chosen))

s1.check_pair_accuracy(pair_accuracy_ref, strict=True)
pair_accuracy = mine_or_ref(globals(), "my_pair_accuracy", ex7_ok, pair_accuracy_ref)

t0 = time.time()
sc0, sr0 = eval_scores(rm, EVAL)
acc0, se0 = pair_accuracy(sc0, sr0)
print(f"untrained reward model on {len(EVAL)} held-out pairs: accuracy {acc0:.3f} ± {se0:.3f} "
      f"| ties {(sc0 == sr0).sum().item()} | {time.time() - t0:.0f}s")

### Reference explanation

### Reference answer and mechanism

The untrained model is at chance: its accuracy is within about two standard errors of 0.5, as in Day 15
(D29). The body understands language, but the random head has no preference. bfloat16 scores can tie
exactly: bfloat16 keeps only 8 significant bits, so two scores near 1.5 can only differ in steps of
0.0078, and in the reference run about one untrained pair in ten tied. Counting a tie as half keeps a
scorer that outputs a constant at exactly 0.5 instead of 0.

### The training step

Each step takes 8 pairs and builds **one batch of 16 conversations**: the 8 prompts with their chosen
answers, then the same 8 prompts with their rejected answers (D27). One forward pass returns 16 scores.
The first 8 are the chosen scores, the last 8 the rejected scores of the same prompts, so row $i$ and row
$i+8$ form pair $i$.

**Precision.** The weights are kept in float32 and only the forward and backward matrix multiplies run
in bfloat16 (`torch.autocast`). Updating bfloat16 weights directly would lose most of the training: a
bfloat16 number has 8 significant bits, so a weight near 0.02 can only change in steps of about
$1.2 \times 10^{-4}$, while an AdamW step at learning rate $10^{-5}$ moves each weight by about $10^{-5}$,
which rounds back to the old value. The reference cell below shows this on one number.

### Prediction

## Exercise 8 · the pairwise loss inside the training loop

Write the loss for one batch: `scores` has shape `[2n]`, chosen block first, then rejected block.
Use your `bt_loss`. One or two lines. The reference cell then runs the training loop with your function.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_rm_pair_loss(scores, n):
    s_chosen, s_rejected = ...    # split the [2n] scores into the chosen and the rejected block
    return ...                    # Bradley–Terry loss of the n pairs

In [ ]:
ex8_ok = s1.check_rm_pair_loss(globals().get("my_rm_pair_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def rm_pair_loss_ref(scores, n):
    return bt_loss_ref(scores[:n], scores[n:])

s1.check_rm_pair_loss(rm_pair_loss_ref, strict=True)
rm_pair_loss = mine_or_ref(globals(), "my_rm_pair_loss", ex8_ok, rm_pair_loss_ref)

def score_batch(model, prompts, answers, max_length=320):
    """Like Q.rm_scores, but with gradients: one forward pass over the whole batch."""
    texts = [Q.render(tok, p, a) for p, a in zip(prompts, answers)]
    enc = tok(texts, return_tensors="pt", padding=True, padding_side="right", truncation=True,
              max_length=max_length).to(DEVICE)
    return model(**enc).logits.squeeze(-1).float()

w = torch.tensor(0.02)
print(f"bf16: 0.02 − 1e-5 == 0.02? {bool((w.bfloat16() - 1e-5).bfloat16() == w.bfloat16())} | "
      f"fp32: {bool(w - 1e-5 == w)}  → the weights stay float32: {rm.dtype}")

BS, LR = 8, 1e-5
opt = torch.optim.AdamW(rm.parameters(), lr=LR, weight_decay=0.0)
rm.train(); rm_losses, t0 = [], time.time()
for i in range(0, len(TRAIN), BS):
    batch = TRAIN[i:i + BS]
    prompts = [p["prompt"] for p in batch] * 2
    with bf16():
        scores = score_batch(rm, prompts, [p["chosen"] for p in batch] + [p["rejected"] for p in batch])
    loss = rm_pair_loss(scores, len(batch))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0)
    opt.step(); opt.zero_grad()
    rm_losses.append(loss.item())
    if len(rm_losses) % 50 == 0:
        print(f"step {len(rm_losses):3d} | mean loss (last 50) {np.mean(rm_losses[-50:]):.3f} | {time.time() - t0:.0f}s")
train_peak = Q.gib(torch.cuda.max_memory_allocated())
del opt; Q.free()
rm.eval(); rm.requires_grad_(False)                  # frozen from here on: it only grades
print(f"trained on {len(TRAIN)} pairs in {time.time() - t0:.0f}s | peak GPU memory so far {train_peak:.1f} GiB")

sc1, sr1 = eval_scores(rm, EVAL)
acc1, se1 = pair_accuracy(sc1, sr1)
diff_se = math.sqrt(se0**2 + se1**2)
print(f"held-out accuracy: {acc0:.3f} ± {se0:.3f} before → {acc1:.3f} ± {se1:.3f} after "
      f"| change {acc1 - acc0:+.3f} ± {diff_se:.3f} (= {(acc1 - acc0) / diff_se:.1f} standard errors)")

### Reference explanation

### Reference answer and mechanism

The split is `scores[:n]` and `scores[n:]` because the batch was built as `chosen + rejected`. An
interleaved split (`scores[0::2]`, `scores[1::2]`) would compare unrelated answers to different prompts and
still produce a plausible-looking loss; the check catches it.

`AdamW(rm.parameters())` updates all ~0.6 B float32 weights, the body and the head, and the clip limits
the gradient norm to 1. In the reference run one pass over 1,200 pairs moved held-out accuracy from
0.52 ± 0.020 to 0.65 ± 0.020, a change of about +0.12 ± 0.028, roughly four standard errors. (The standard
error of a difference is taken as $\sqrt{se_0^2 + se_1^2}$, which is conservative because the two
measurements share the same pairs.) bfloat16 GPU kernels are not bitwise deterministic, so your numbers
can differ from these by a point or two. Day 15's 3,000-pair run reached a similar range; with label
noise like HH's, the attainable accuracy is well below 100% (D3).

An earlier version of this notebook updated bfloat16 weights directly, as Day 15 · 04 does. Measured
after one step, only 9.7% of the parameters changed, against 74% with float32 master weights, yet that
version still reached 0.63. The difference from 0.65 is within noise, so the float32 fix is about the
update doing what the optimizer computed, not a measured accuracy gain.

In [ ]:
gaps = (sc1 - sr1)
p_sym = torch.sigmoid(torch.cat([gaps, -gaps]))        # each pair in both orders: label 1, then label 0
y_sym = torch.cat([torch.ones_like(gaps), torch.zeros_like(gaps)])
rel_rm = reliability(p_sym, y_sym, n_bins=10)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
smooth = np.convolve(rm_losses, np.ones(20) / 20, mode="valid")
axes[0].plot(rm_losses, color=GRAY, alpha=0.5, lw=0.8, label="per step (8 pairs)")
axes[0].plot(np.arange(len(smooth)) + 10, smooth, color=BLUE, label="20-step mean")
axes[0].axhline(math.log(2), color=RED, ls="--", lw=1, label="log 2 (always 50/50)")
axes[0].set_xlabel("step"); axes[0].set_ylabel("pairwise loss"); axes[0].set_title("Reward-model training")
axes[0].legend(fontsize=8)
axes[1].hist(gaps.numpy(), bins=40, color=BLUE, edgecolor="white"); axes[1].axvline(0, color="black", lw=1)
axes[1].set_xlabel("r(chosen) − r(rejected), held-out")
axes[1].set_title(f"Held-out gaps: accuracy {acc1:.3f} ± {se1:.3f}")
axes[2].plot(rel_rm["conf"], rel_rm["freq"], marker="o", color=BLUE, label=f"trained RM, ECE {rel_rm['ece']:.3f}")
for c, f, n in zip(rel_rm["conf"], rel_rm["freq"], rel_rm["count"]):
    axes[2].annotate(str(n), (c, f), fontsize=7, color=GRAY, xytext=(3, -9), textcoords="offset points")
axes[2].plot([0, 1], [0, 1], color=GRAY, lw=0.8, ls=":")
axes[2].set_xlabel("predicted P(first answer preferred)"); axes[2].set_ylabel("observed frequency")
axes[2].set_title("Reliability (grey numbers: predictions per bin)"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("bins (conf, freq, count):", [(round(c, 2), round(f, 2), n) for c, f, n in zip(rel_rm["conf"], rel_rm["freq"], rel_rm["count"])])

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Qwen3 reward-model training loss, held-out score gaps and the reliability diagram](../figures/chapter-10/study-path-s1_reward_models-03.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

**Reading the figure.** The training loss is a weak signal here. Each step sees only 8 pairs, so the
per-step loss jumps between about 0.3 and 2, and in the reference runs the 20-step mean stays at or
above $\log 2$ for roughly the first 100 steps: early updates make some gaps large in the wrong
direction, and a wrong gap costs more than $\log 2$. It drops below $\log 2$ only in the last few dozen
steps, and it would stay far above 0 even with much more data, as Part B predicts for noisy labels. The
held-out accuracy, measured with a standard error, is the evidence that the model learned something. The gap histogram has a large mass near 0, where the model is unsure, and a tail of
confidently wrong pairs. For the reliability diagram each held-out pair is scored in both orders
(prediction $\sigma(\Delta)$ with label 1 and $\sigma(-\Delta)$ with label 0), so the diagram is symmetric about
(0.5, 0.5) by construction. Bins with few predictions (grey counts) have large sampling error, and a
deviation from the diagonal there is not strong evidence of miscalibration. In the reference run the
middle bins (predictions 0.2–0.8) lie within about 0.1 of the diagonal, while the outer bins are
**over-confident**: pairs predicted at 0.96 were preferred only about 75% of the time (about 90
predictions), and the ECE is about 0.08.
Part B showed the mechanism that should make a model trained on noisy labels *under*-confident; here
the extreme scores come from pairs the model ranks confidently and wrongly, and one pass over 1,200
pairs is too little data to pull them back. One run with 600 pairs reads calibration only coarsely.

---

# Part E · Length audit

In HH-RLHF helpful-base the chosen answer is longer than the rejected one more often than not (Day 15 ·
04 Predict 3). A reward model trained on these labels can therefore lower its loss by rewarding length,
whether or not length is what the judges valued. Accuracy alone can't tell the two apart. We audit in
two ways: correlation between score and length, and accuracy on the pairs where length and the label
disagree.

There are two correlations to compute, and they answer different questions. Across **all** answers, the
raw score mixes in a per-prompt offset $c(x)$, which pairwise training never constrains (the shift
symmetry of Part A). Within a **pair**, the offset cancels: the score gap against the word-count gap
asks whether, for the same prompt, the longer answer gets the higher score.

The **Pearson correlation** of two vectors $a, b$ is

$$\rho = \frac{\sum_i (a_i - \bar a)(b_i - \bar b)}{\sqrt{\sum_i (a_i - \bar a)^2}\,\sqrt{\sum_i (b_i - \bar b)^2}}.$$

### Prediction

## Exercise 9 · the Pearson correlation

Write $\rho$ for two 1-D tensors. Two lines: centre both, then the formula.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_pearson(a, b):
    a, b = a.double(), b.double()
    a, b = ...                # subtract each vector's mean
    return ...                # Σ a·b / (‖a‖·‖b‖), as a float

In [ ]:
ex9_ok = s1.check_pearson(globals().get("my_pearson"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def pearson_ref(a, b):
    a, b = a.double() - a.double().mean(), b.double() - b.double().mean()
    return float((a * b).sum() / (a.norm() * b.norm()))

s1.check_pearson(pearson_ref, strict=True)
pearson = mine_or_ref(globals(), "my_pearson", ex9_ok, pearson_ref)

def words(text):
    return len(text.split())

len_c = torch.tensor([words(p["chosen"]) for p in EVAL])
len_r = torch.tensor([words(p["rejected"]) for p in EVAL])
print(f"held-out pairs: chosen longer {(len_c > len_r).float().mean():.1%} | shorter {(len_c < len_r).float().mean():.1%} "
      f"| equal {(len_c == len_r).float().mean():.1%} | median words {len_c.median().item()} vs {len_r.median().item()}")

### Reference explanation

### Reference answer and mechanism

Without centring, the formula computes the cosine similarity of the raw vectors, which is large whenever
both vectors have large means (word counts are always positive). Centring removes that. A correlation
measured on $N$ points has a standard error of roughly $1/\sqrt N$ near 0: about 0.03 for 1,200 answers
and 0.04 for 600 pairs, so values within ±0.08 of each other are not distinguishable here.

### Prediction

## Predict 3 · does the untrained model already follow length?

Before training, the score head is random. Do you expect the **untrained** reward model's scores to be
correlated with answer length? Consider both the raw score across all 1,200 answers and the
within-pair gap against the word-count gap. After training, which of the two correlations should
grow, and how large will it get?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
corr = {}
for name, (sc, sr) in {"untrained": (sc0, sr0), "trained": (sc1, sr1)}.items():
    corr[name, "raw"] = pearson(torch.cat([sc, sr]), torch.cat([len_c, len_r]))
    corr[name, "pair"] = pearson(sc - sr, len_c - len_r)
    print(f"{name:9s} | raw score vs words (1,200 answers): {corr[name, 'raw']:+.3f} "
          f"| score gap vs word gap (600 pairs): {corr[name, 'pair']:+.3f}")

### Reference explanation

### Reference answer and mechanism

Read the signs and sizes from your run; the reference run's values are stated here.

**Untrained.** In the reference run the raw-score correlation is +0.09 and the within-pair one +0.03,
both small (standard errors about 0.03 and 0.04). A random linear readout of a pretrained hidden state
is not guaranteed to be unrelated to length: the `<|im_end|>` state of a long conversation differs
systematically from that of a short one, and a random direction $w$ picks up some of that difference,
with a sign set by the random draw of $w$. Exercise 10 shows this draw matters more than the
correlations suggest.

**Trained.** Training pushes the gap toward the labels, and in this data the chosen answer is the longer
one in 59% of the held-out pairs (38% shorter). The reference run reaches about +0.44 (raw) and **+0.56**
(within pair), higher than Day 15's +0.4 after 3,000 pairs. This doesn't prove that the model rewards
length *for its own sake*: longer HH answers are often more complete, and a correlation can't separate the
two. The next exercise holds length against the label.

### Prediction

## Exercise 10 · length-controlled accuracy

Split the pairs by length: pairs where the chosen answer has **more** words than the rejected one, and pairs where it
has **fewer** (pairs of equal length go in neither). Return `pair_accuracy` on each slice plus the slice
size. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_length_sliced_accuracy(s_c, s_r, len_c, len_r):
    longer, shorter = ...         # boolean masks over pairs: chosen longer / chosen shorter
    return {"chosen_longer": pair_accuracy(s_c[longer], s_r[longer]) + (int(longer.sum()),),
            "chosen_shorter": ...}    # same for the other slice

In [ ]:
ex10_ok = s1.check_length_sliced_accuracy(globals().get("my_length_sliced_accuracy"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def length_sliced_accuracy_ref(s_c, s_r, len_c, len_r):
    longer, shorter = len_c > len_r, len_c < len_r
    return {"chosen_longer": pair_accuracy_ref(s_c[longer], s_r[longer]) + (int(longer.sum()),),
            "chosen_shorter": pair_accuracy_ref(s_c[shorter], s_r[shorter]) + (int(shorter.sum()),)}

s1.check_length_sliced_accuracy(length_sliced_accuracy_ref, strict=True)
length_sliced_accuracy = mine_or_ref(globals(), "my_length_sliced_accuracy", ex10_ok, length_sliced_accuracy_ref)

sliced = {name: length_sliced_accuracy(sc, sr, len_c, len_r)
          for name, (sc, sr) in {"untrained": (sc0, sr0), "trained": (sc1, sr1)}.items()}
for name, res in sliced.items():
    print(name, " | ".join(f"{k}: {a:.3f} ± {s:.3f} (n = {n})" for k, (a, s, n) in res.items()))
(al, sl, _), (as_, ss, _) = sliced["trained"]["chosen_longer"], sliced["trained"]["chosen_shorter"]
print(f"trained: longer-minus-shorter accuracy gap {al - as_:+.3f} ± {math.sqrt(sl**2 + ss**2):.3f}")

### Reference explanation

### Reference answer and mechanism

This is the audit that matters. On the pairs where the chosen answer is **shorter**, length points to the
wrong answer, so a model that relied mainly on length would score *below* 50% there.

Reference run (600 held-out pairs; 355 chosen-longer, 228 chosen-shorter):

| | chosen longer | chosen shorter |
|---|---|---|
| untrained | 0.59 ± 0.03 | 0.41 ± 0.03 |
| trained | 0.79 ± 0.02 | **0.43 ± 0.03** |

Two findings. First, the **untrained** model already splits by length: its random head ranks the longer
answer higher in about 59% of pairs, regardless of quality. Its within-pair Pearson correlation was
only about +0.03, because Pearson measures how gap *sizes* line up, while the slices measure only the
*sign*. An audit needs the untrained baseline.

Second, **training improved only the chosen-longer slice**: +0.20 there, while on the chosen-shorter
pairs accuracy stayed below 0.5 (about two standard errors under chance) and moved by no more than its
noise. So the overall gain from 0.52 to 0.65 comes from pairs where the preferred answer is also the
longer one; where length and the human label disagree, this model is no better than its untrained
version. Whether it sides with length *more* often than chance can't be settled with 228 pairs. In
earlier runs of this notebook with bfloat16 weights this slice landed between 0.33 and 0.36, so the exact
value moves between runs; "no gain where length disagrees" held in all of them.

This doesn't show that the model computes word counts; it shows what the scores agree with on this data.
It is exactly the regularity a policy optimized against this reward can exploit by writing longer
answers (stage S7, Chapter 14), and it is Part B's length-biased judge seen from the outside.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
gap_words = (len_c - len_r).numpy()
axes[0].scatter(np.clip(gap_words, -200, 200), (sc1 - sr1).numpy(), s=8, color=BLUE, alpha=0.5)
axes[0].axhline(0, color=GRAY, lw=0.8); axes[0].axvline(0, color=GRAY, lw=0.8)
axes[0].set_xlabel("words(chosen) − words(rejected), clipped at ±200"); axes[0].set_ylabel("score gap (trained)")
axes[0].set_title(f"Within-pair: ρ = {corr['trained', 'pair']:+.2f} (untrained {corr['untrained', 'pair']:+.2f})")
x = np.arange(2)
for off, name, c in ((-0.18, "untrained", GRAY), (0.18, "trained", BLUE)):
    vals = [sliced[name][k] for k in ("chosen_longer", "chosen_shorter")]
    axes[1].bar(x + off, [v[0] for v in vals], width=0.36, color=c, yerr=[2 * v[1] for v in vals], capsize=4,
                label=f"{name} (bars: ±2 SE)")
axes[1].axhline(0.5, color=RED, ls="--", lw=1)
axes[1].set_xticks(x, [f"chosen longer (n = {sliced['trained']['chosen_longer'][2]})",
                       f"chosen shorter (n = {sliced['trained']['chosen_shorter'][2]})"])
axes[1].set_ylabel("held-out pair accuracy"); axes[1].set_ylim(0, 1); axes[1].legend(fontsize=8)
axes[1].set_title("Accuracy when length agrees vs disagrees with the label")
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Score gap against word-count gap, and pair accuracy sliced by whether the chosen answer is longer](../figures/chapter-10/study-path-s1_reward_models-04.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part F · Outcome labels and process labels

So far every label has been one judgment of a **whole** response, an **outcome label**. For multi-step
reasoning there is a second option: a **process label** judges each intermediate step. Lightman et al.
(2023, "Let's Verify Step by Step") had human labellers rate every step of model-written solutions to
MATH problems (the PRM800K dataset) and trained a **process reward model** (PRM): a network that outputs
$p_t = P(\text{step } t \text{ is correct} \mid \text{problem}, \text{steps } 1..t)$, read at the end of each
step, the same per-position readout as Part C but with a label at each step end. They scored a whole
solution by the product $\prod_t p_t$ and found that, for picking the best of many sampled solutions,
process supervision beat outcome supervision. Uesato et al. (2022) had compared the two kinds of
feedback on grade-school math earlier.

This part is a toy: two word problems, seven **authored** solutions with deliberate errors, an exact
rule that labels each step, and a declared stand-in for a trained PRM. It shows what each kind of label
can and cannot see; it measures nothing about real PRMs. Each step is a tuple `(a, op, b, c)` meaning
"$a \;\text{op}\; b = c$".

In [ ]:
for key, prob in s1.PROBLEMS.items():
    print(f"[{key}] {prob['text']}  (numbers {prob['numbers']}, answer {prob['gold']})")
print()
for name, key, steps in s1.SOLUTIONS:
    print(f"{name:12s}: " + "  →  ".join(f"{a} {op} {b} = {c}" for a, op, b, c in steps))

### Prediction

## Exercise 11 · outcome labels and process labels

Write the two labelling rules.

- `my_outcome_label(steps, gold)`: 1 if the final result (the `c` of the last step) equals `gold`, else 0.
- `my_process_labels(numbers, steps)`: one 0/1 label per step. A step is correct when its arithmetic is
  right, applying `OPS[op]` to `a` and `b` gives `c`, **and** both operands are already known: given in the problem or computed by
  an earlier step. The step's result becomes known even if it is wrong, so each error is charged once,
  at the step that makes it.

One line each.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
OPS = s1.OPS                                   # {"+": operator.add, "-": operator.sub, "*": operator.mul}

def my_outcome_label(steps, gold):
    return ...                                 # 1 if the last step's result equals gold, else 0

def my_process_labels(numbers, steps):
    known, labels = set(numbers), []
    for a, op, b, c in steps:
        labels.append(...)                     # 1 if arithmetic right and a, b both known, else 0
        known.add(c)
    return labels

In [ ]:
ex11_ok = s1.check_labels(globals().get("my_outcome_label"), globals().get("my_process_labels"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
OPS = s1.OPS

def outcome_label_ref(steps, gold):
    return int(steps[-1][3] == gold)

def process_labels_ref(numbers, steps):
    known, labels = set(numbers), []
    for a, op, b, c in steps:
        labels.append(int(OPS[op](a, b) == c and a in known and b in known))
        known.add(c)
    return labels

s1.check_labels(outcome_label_ref, process_labels_ref, strict=True)
outcome_label = mine_or_ref(globals(), "my_outcome_label", ex11_ok, outcome_label_ref)
process_labels = mine_or_ref(globals(), "my_process_labels", ex11_ok, process_labels_ref)

labels = {}
print("solution      outcome   process labels   first wrong step")
for name, key, steps in s1.SOLUTIONS:
    prob = s1.PROBLEMS[key]
    o, p = outcome_label(steps, prob["gold"]), process_labels(prob["numbers"], steps)
    labels[name] = (o, p)
    first = next((i + 1 for i, l in enumerate(p) if l == 0), None)
    print(f"{name:12s}  {o}         {str(p):16s} {first if first else '—'}")
assert labels["apples · C"] == (1, [1, 0, 0])

### Reference explanation

### Reference answer and mechanism

The table shows the gap between the two kinds of label:

- **apples · C** reaches the right answer, 29, through two wrong steps ($36 - 15 = 22$, then $22 + 8 = 29$).
  Its outcome label is 1, so outcome supervision would *reward* this reasoning. The process labels mark
  steps 2 and 3 as wrong.
- **apples · B** and **apples · D** both get outcome 0, but for different reasons: an arithmetic slip
  at step 2, and a number (35) that nothing computed. One outcome bit can't distinguish them; the
  process labels put the 0 at step 2 in both, and the rule says why.
- After a wrong step, later steps that compute correctly *from the wrong number* get label 1
  (apples · B step 3: $11 + 8 = 19$ is right arithmetic). The error is charged where it is made.

An outcome label needs only the gold answer, so it is cheap and exactly checkable, which is why
verifiable outcome rewards (stage S5) are attractive. Process labels need a judge for every step,
which Lightman et al. paid human labellers for.

### From step scores to a solution score

A PRM outputs one probability per step. To rank whole solutions we must **aggregate** them. Two common
choices: the **product** $\prod_t p_t$, which is the probability that every step is correct if the steps'
errors were independent (Lightman et al.'s choice), and the **minimum** $\min_t p_t$, the weakest step. The
step with the lowest $p_t$ **localises** the most likely error.

We don't have a trained PRM, so we use a declared stand-in, `s1.simulated_prm`: it returns 0.92 for a step
labelled correct and 0.15 for a step labelled wrong. It ranks every step correctly, with fixed confidence,
which isolates what the aggregation does.

### Prediction

## Exercise 12 · aggregate step scores

Return the minimum, the product and the index of the weakest step for a `[K]` tensor of step probabilities.
Three short expressions.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_aggregate(step_probs):
    return {"min": ...,                       # weakest step's probability (a float)
            "product": ...,                   # product of the step probabilities (a float)
            "weakest_step": ...}              # 0-based index of the lowest probability (an int)

In [ ]:
ex12_ok = s1.check_aggregate(globals().get("my_aggregate"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def aggregate_ref(step_probs):
    return {"min": step_probs.min().item(), "product": step_probs.prod().item(),
            "weakest_step": int(step_probs.argmin())}

s1.check_aggregate(aggregate_ref, strict=True)
aggregate = mine_or_ref(globals(), "my_aggregate", ex12_ok, aggregate_ref)
print(inspect.getsource(s1.simulated_prm))

agg = {}
print("solution      outcome  steps  PRM min  PRM product  weakest step")
for name, key, steps in s1.SOLUTIONS:
    o, p = labels[name]
    agg[name] = aggregate(s1.simulated_prm(p))
    weakest = agg[name]["weakest_step"] + 1 if agg[name]["min"] < 0.5 else "— (no step below 0.5)"
    print(f"{name:12s}  {o}        {len(steps)}      {agg[name]['min']:.2f}     {agg[name]['product']:.3f}        {weakest}")

### Reference explanation

### Reference answer and mechanism

Both aggregations put apples · C, the right answer by wrong steps, far below the correct solutions, and
`weakest_step` points at its first wrong step (step 2). With two equally bad steps, `argmin` returns the
first, which is the step where the solution went wrong. An outcome scorer can only say "1" for apples · C and
"0" for the other two wrong solutions, with no location.

The two aggregations differ on wrong solutions with several bad steps: the product multiplies the two
0.15s of apples · C (0.021), while the minimum counts the weakest step once (0.15). The next prediction
looks at how they differ on correct solutions.

### Prediction

## Predict 4 · which aggregation prefers shorter solutions?

tickets · A (3 steps) and tickets · B (4 steps) are both entirely correct. Under the stand-in PRM, which
gets the higher **product** score, and which the higher **min** score? If a policy were trained to
maximize the product score of its own solutions, what would that push it to do?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
for name in ("tickets · A", "tickets · B"):
    print(f"{name}: steps {len(labels[name][1])} | min {agg[name]['min']:.3f} | product {agg[name]['product']:.4f}")
print(f"0.92³ = {0.92**3:.4f}, 0.92⁴ = {0.92**4:.4f}")
assert agg["tickets · A"]["min"] == agg["tickets · B"]["min"]
assert agg["tickets · A"]["product"] > agg["tickets · B"]["product"]

### Reference explanation

### Reference answer and mechanism

The **minimum** gives both 0.92: it doesn't care how many correct steps there are. The **product** gives
tickets · A $0.92^3 = 0.779$ and tickets · B $0.92^4 = 0.716$: every extra correct step multiplies in
another factor below 1, so a longer correct solution scores lower. A policy optimized against the product
would be pushed toward **fewer, larger steps**, the opposite of the length bias in Part E but the same
kind of problem: the aggregation rule, not correctness, decides part of the reward. The size of the effect
depends on how close the PRM's confidence in correct steps is to 1. Real PRMs are not this tidy, and the
choice between product and minimum is an empirical one.

In [ ]:
names = [n for n, _, _ in s1.SOLUTIONS]
K = max(len(s) for _, _, s in s1.SOLUTIONS)
grid = np.full((len(names), K), np.nan)
for i, name in enumerate(names):
    probs = s1.simulated_prm(labels[name][1]).numpy()
    grid[i, :len(probs)] = probs
fig, ax = plt.subplots(figsize=(10, 3.8))
cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list("rg", [RED, "#FDE68A", GREEN])
ax.imshow(grid, cmap=cmap, vmin=0, vmax=1, aspect="auto")
for i, (name, key, steps) in enumerate(s1.SOLUTIONS):
    for t, (a, op, b, c) in enumerate(steps):
        ax.text(t, i, f"{a} {op} {b} = {c}\np = {grid[i, t]:.2f}", ha="center", va="center", fontsize=7.5)
    o = labels[name][0]
    ax.text(K - 0.35, i, f"outcome {o} | min {agg[name]['min']:.2f} | prod {agg[name]['product']:.3f}",
            ha="left", va="center", fontsize=8, color=GREEN if o else RED)
ax.set_yticks(range(len(names)), names); ax.set_xticks(range(K), [f"step {t + 1}" for t in range(K)])
ax.set_xlim(-0.5, K + 1.9)
ax.set_title("Process labels locate the error; the outcome label gives one bit per solution (toy, authored data)")
for s in ("left", "bottom"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Per-step scores of seven authored solutions with their outcome labels and aggregated scores](../figures/chapter-10/study-path-s1_reward_models-05.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

In [ ]:
peak = Q.gib(torch.cuda.max_memory_allocated())
minutes = (time.time() - T_START) / 60
print(f"total run time {minutes:.1f} min | peak GPU memory {peak:.1f} GiB")
print(f"Qwen3 RM: held-out accuracy {acc0:.3f} ± {se0:.3f} → {acc1:.3f} ± {se1:.3f} on {len(EVAL)} pairs")
print("length slices (trained):", {k: (round(a, 3), round(s, 3), n) for k, (a, s, n) in sliced["trained"].items()})
del rm; Q.free()

---

## What you built and measured

| Part | You wrote | What the run showed |
|---|---|---|
| A · Bradley–Terry | `bt_prob`, `bt_loss` (stable), the gradient by hand | gradient $-(1-\sigma(\Delta))$ matches autograd exactly; a shift changes nothing, a scale changes confidence |
| B · Linear RM | the training-step loss, the noisy-label probability, the reliability table | clean labels recover $w^* = (2, 0.5)$; 20% random flips shrink $\|w\|$ to about a third but keep its direction (99.9% ranking agreement) and leave the model under-confident against clean labels; a length-biased judge changing about 8% of labels doubles $w_\text{length}/w_\text{help}$ and costs about 8 points of ranking agreement |
| C · Pooling | the last non-pad index | your index reproduces Hugging Face's score for left and right padding; per-position outputs are identical before a changed token and differ from it onward (causal attention) |
| D · Qwen3 RM | pair accuracy with SE, the pairwise loss in the loop | held-out accuracy about 0.52 → 0.65 (± 0.02 each) after 1,200 pairs, roughly four standard errors; calibrated in the middle, over-confident in the outer bins; float32 master weights (bf16 weights changed only 9.7% of parameters per step) |
| E · Length audit | Pearson correlation, length-sliced accuracy | within-pair length correlation about +0.03 → +0.56; accuracy 0.59 → 0.79 when the chosen answer is longer, but 0.41 → 0.43 (below chance, no gain) when it is shorter |
| F · Outcome vs process | both labelling rules, min/product/argmin | outcome labels reward a right answer reached by wrong steps; process labels and `argmin` locate the first error; the product penalizes extra correct steps |

**Evidence.** The toy results in Parts A, B and F are exact or come from seeded float64 simulations with
a known truth, so the statements about them are checked by assertions. The Qwen3 results come from **one**
training run (one seed, 1,200 pairs, one pass) evaluated on 600 held-out HH pairs: the accuracy change
and the length-slice gap are reported with standard errors and are several standard errors in the
reference run (the chosen-shorter slice itself is only about two below chance), but a different seed or sample would move each by a few points. The run doesn't
establish how this reward model behaves under optimization pressure, on other data, or at larger scale,
and the length audit shows a correlation with length on this data, not a mechanism. Part F's PRM is a
declared stand-in, so it shows what the aggregation rules do, not how well real process reward models
work.

**Next stage.** [S2 · The KL-regularized optimum and DPO](s2_kl_optimum_and_dpo.ipynb): the reward model
you just trained is a proxy; S2 derives what policy maximizes "reward minus a KL penalty to the
reference", and how DPO learns that policy from the same preference pairs without training a reward
model at all.